# Kon-Tiki Biochar Volume — Web Dashboard (upload video → volume)

**One step:** `Runtime → Change runtime type → GPU (T4)`, then **Run this cell**.
After ~1–2 min it prints a **public link** (`…gradio.live`). Open it → **upload a kiln
video → get the volume**. No download / re-upload. Share the link with anyone.

_The GPU reconstruction runs here on Colab's free GPU (this is the only part that needs a
GPU); the volume maths is the same code validated to ~2–4% on ground truth._


In [ ]:
import os, sys, glob, shutil, base64, subprocess, torch, numpy as np, cv2
subprocess.run("pip -q install gradio opencv-python-headless scipy", shell=True)
if not os.path.exists("vggt"):
    subprocess.run("git clone -q https://github.com/facebookresearch/vggt.git", shell=True)
subprocess.run("grep -viE '^(torch|torchvision|torchaudio|numpy)' vggt/requirements.txt > /tmp/r.txt", shell=True)
subprocess.run("pip -q install -r /tmp/r.txt", shell=True)
if "vggt" not in sys.path: sys.path.append("vggt")
assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > GPU (T4), then re-run."

open("estimate_volume.py", "w", encoding="utf-8").write(base64.b64decode("...").decode("utf-8"))
open("aruco_tools.py", "w", encoding="utf-8").write(base64.b64decode("IiIiQXJVY28gbWFya2VyIHRvb2xzIGZvciBNRVRSSUMgU0NBTEUgKHRoZSBkb2N1bWVudCdzIG1ldGhvZCkuCgpUaGUgMy1EIHJlY29uc3RydWN0aW9uIGhhcyBubyByZWFsIHNpemUgb24gaXRzIG93bi4gV2UgcHV0IHByaW50ZWQgQXJVY28gbWFya2VycyBvZgphIEtOT1dOIHNpemUgaW4gdGhlIHNjZW5lOyB0aGlzIG1vZHVsZSAoYSkgZ2VuZXJhdGVzIHByaW50YWJsZSBtYXJrZXJzLCBhbmQgKGIpIHJlYWRzCnRoZSB0cnVlIHNjYWxlIChjbSBwZXIgcmVjb25zdHJ1Y3Rpb24tdW5pdCkgZnJvbSB0aGUgbWFya2VycyB1c2luZyB0aGUgcGVyLXBpeGVsIDMtRApwb2ludHMgdGhlIHJlY29uc3RydWN0b3IgcHJvZHVjZXMg4oCUIHNvIHRoZSBzaXplIGlzIG1lYXN1cmVkIGZyb20gdGhlIHZpZGVvLCBub3QgYXNzdW1lZC4KClNlbGYtdGVzdDogIHB5dGhvbiBhcnVjb190b29scy5weQoiIiIKaW1wb3J0IG9zLCBudW1weSBhcyBucCwgY3YyCgpESUNUID0gIkRJQ1RfNFg0XzUwIgoKCmRlZiBfZGljdChuYW1lPURJQ1QpOgogICAgcmV0dXJuIGN2Mi5hcnVjby5nZXRQcmVkZWZpbmVkRGljdGlvbmFyeShnZXRhdHRyKGN2Mi5hcnVjbywgbmFtZSkpCgoKZGVmIGdlbmVyYXRlX21hcmtlcnMoaWRzPSgwLCAxLCAyLCAzKSwgb3V0X2Rpcj0ibWFya2VycyIsIHB4PTcwMCwgYm9yZGVyPTgwLCBuYW1lPURJQ1QpOgogICAgIiIiU2F2ZSBwcmludGFibGUgbWFya2VyIFBOR3MgKHdoaXRlIGJvcmRlciBrZXB0KS4gUHJpbnQsIE1FQVNVUkUgdGhlIGJsYWNrCiAgICBzcXVhcmUncyByZWFsIGVkZ2UsIGFuZCBwYXNzIHRoYXQgbWVhc3VyZWQgY20gdG8gc2NhbGVfZnJvbV9tYXJrZXIoKS4iIiIKICAgIG9zLm1ha2VkaXJzKG91dF9kaXIsIGV4aXN0X29rPVRydWUpCiAgICBkID0gX2RpY3QobmFtZSk7IHBhdGhzID0gW10KICAgIGZvciBtaWQgaW4gaWRzOgogICAgICAgIG0gPSBjdjIuYXJ1Y28uZ2VuZXJhdGVJbWFnZU1hcmtlcihkLCBtaWQsIHB4KQogICAgICAgIGNhbnZhcyA9IG5wLmZ1bGwoKHB4ICsgMiAqIGJvcmRlciwgcHggKyAyICogYm9yZGVyKSwgMjU1LCBucC51aW50OCkKICAgICAgICBjYW52YXNbYm9yZGVyOmJvcmRlciArIHB4LCBib3JkZXI6Ym9yZGVyICsgcHhdID0gbQogICAgICAgIGN2Mi5wdXRUZXh0KGNhbnZhcywgZiJ7bmFtZX0gaWQ9e21pZH0iLCAoYm9yZGVyLCBib3JkZXIgLSAyNSksCiAgICAgICAgICAgICAgICAgICAgY3YyLkZPTlRfSEVSU0hFWV9TSU1QTEVYLCAwLjksIDAsIDIsIGN2Mi5MSU5FX0FBKQogICAgICAgIHAgPSBvcy5wYXRoLmpvaW4ob3V0X2RpciwgZiJhcnVjb197bWlkfS5wbmciKTsgY3YyLmltd3JpdGUocCwgY2FudmFzKTsgcGF0aHMuYXBwZW5kKHApCiAgICByZXR1cm4gcGF0aHMKCgpkZWYgc2NhbGVfZnJvbV9tYXJrZXIoaW1hZ2VzLCB3b3JsZF9wb2ludHMsIG1hcmtlcl9jbSwgbmFtZT1ESUNULCBkZWJ1Zz1GYWxzZSk6CiAgICAiIiJpbWFnZXM6IGxpc3Qgb2YgZnJhbWVzIGF0IHRoZSBTQU1FIHJlc29sdXRpb24gYXMgd29ybGRfcG9pbnRzICh0aGUgaW1hZ2VzIGZlZAogICAgdG8gdGhlIHJlY29uc3RydWN0b3IpLiB3b3JsZF9wb2ludHM6IGFycmF5IFtOLCBILCBXLCAzXSBvZiBwZXItcGl4ZWwgMy1EIHBvc2l0aW9ucy4KICAgIG1hcmtlcl9jbTogdGhlIE1FQVNVUkVEIHJlYWwgZWRnZSBsZW5ndGggb2YgdGhlIHByaW50ZWQgbWFya2VyIChjbSkuCiAgICBSZXR1cm5zIGRpY3Qoc2NhbGVfY21fcGVyX3VuaXQsIG4sIHNwcmVhZF9wY3QpIG9yIE5vbmUgaWYgbm8gbWFya2VyIHNlZW4uIiIiCiAgICBkID0gX2RpY3QobmFtZSkKICAgIGRldCA9IGN2Mi5hcnVjby5BcnVjb0RldGVjdG9yKGQsIGN2Mi5hcnVjby5EZXRlY3RvclBhcmFtZXRlcnMoKSkKICAgIHdvcmxkX3BvaW50cyA9IG5wLmFzYXJyYXkod29ybGRfcG9pbnRzKQogICAgc2NhbGVzID0gW10KICAgIGZvciBpIGluIHJhbmdlKGxlbihpbWFnZXMpKToKICAgICAgICBpbWcgPSBpbWFnZXNbaV0KICAgICAgICBncmF5ID0gaW1nIGlmIChpbWcubmRpbSA9PSAyKSBlbHNlIGN2Mi5jdnRDb2xvcihpbWcsIGN2Mi5DT0xPUl9SR0IyR1JBWSkKICAgICAgICBjb3JuZXJzLCBpZHMsIF8gPSBkZXQuZGV0ZWN0TWFya2VycyhncmF5KQogICAgICAgIGlmIGlkcyBpcyBOb25lOgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIHdwID0gd29ybGRfcG9pbnRzW2ldOyBILCBXID0gd3Auc2hhcGVbOjJdCiAgICAgICAgZm9yIGMgaW4gY29ybmVyczoKICAgICAgICAgICAgeHkgPSBucC5yb3VuZChjWzBdKS5hc3R5cGUoaW50KQogICAgICAgICAgICB4eVs6LCAwXSA9IG5wLmNsaXAoeHlbOiwgMF0sIDAsIFcgLSAxKTsgeHlbOiwgMV0gPSBucC5jbGlwKHh5WzosIDFdLCAwLCBIIC0gMSkKICAgICAgICAgICAgUDMgPSB3cFt4eVs6LCAxXSwgeHlbOiwgMF1dICAgICAgICAgICAgICAgICAgICAgIyA0IGNvcm5lciAzLUQgcG9zaXRpb25zCiAgICAgICAgICAgIGlmIG5vdCBucC5pc2Zpbml0ZShQMykuYWxsKCk6CiAgICAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICAgICBlZGdlcyA9IFtucC5saW5hbGcubm9ybShQM1soayArIDEpICUgNF0gLSBQM1trXSkgZm9yIGsgaW4gcmFuZ2UoNCldCiAgICAgICAgICAgIGUgPSBmbG9hdChucC5tZWRpYW4oZWRnZXMpKQogICAgICAgICAgICBpZiBlID4gMWUtOToKICAgICAgICAgICAgICAgIHNjYWxlcy5hcHBlbmQobWFya2VyX2NtIC8gZSkKICAgIGlmIG5vdCBzY2FsZXM6CiAgICAgICAgcmV0dXJuIE5vbmUKICAgIHNjYWxlcyA9IG5wLmFycmF5KHNjYWxlcyk7IG1lZCA9IGZsb2F0KG5wLm1lZGlhbihzY2FsZXMpKQogICAgcmVzID0gZGljdChzY2FsZV9jbV9wZXJfdW5pdD1tZWQsIG49bGVuKHNjYWxlcyksCiAgICAgICAgICAgICAgIHNwcmVhZF9wY3Q9ZmxvYXQoMTAwICogbnAuc3RkKHNjYWxlcykgLyBtYXgobWVkLCAxZS05KSkpCiAgICBpZiBkZWJ1ZzoKICAgICAgICBwcmludCgiICBbYXJ1Y29dIiwgcmVzKQogICAgcmV0dXJuIHJlcwoKCmRlZiBfc2VsZnRlc3QoKToKICAgICIiIlZlcmlmeSBkZXRlY3Rpb24gKyB0aGUgc2NhbGUgbWF0aCBvbiBhIHN5bnRoZXRpYyBmbGF0IHBsYW5lIG9mIEtOT1dOIHNjYWxlLiIiIgogICAgcHgsIGJyZCA9IDI0MCwgNjAKICAgIG0gPSBjdjIuYXJ1Y28uZ2VuZXJhdGVJbWFnZU1hcmtlcihfZGljdCgpLCAwLCBweCkKICAgIGltZyA9IG5wLmZ1bGwoKHB4ICsgMiAqIGJyZCwgcHggKyAyICogYnJkKSwgMjU1LCBucC51aW50OCkKICAgIGltZ1ticmQ6YnJkICsgcHgsIGJyZDpicmQgKyBweF0gPSBtICAgICAgICAgICAgICAgICAgICAjIG1hcmtlciBzcGFucyBweCBwaXhlbHMKICAgIEhpbWcsIFdpbWcgPSBpbWcuc2hhcGUKICAgIGsgPSAwLjUgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIyB1bml0cyBwZXIgcGl4ZWwgKHN5bnRoZXRpYykKICAgIHhzID0gbnAuYXJhbmdlKFdpbWcpICogazsgeXMgPSBucC5hcmFuZ2UoSGltZykgKiBrCiAgICBYWCwgWVkgPSBucC5tZXNoZ3JpZCh4cywgeXMpCiAgICB3cCA9IG5wLnN0YWNrKFtYWCwgWVksIG5wLnplcm9zX2xpa2UoWFgpXSwgLTEpW05vbmVdICAgICMgWzEsSCxXLDNdCiAgICBtYXJrZXJfY20gPSAyMC4wICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICMgcHJldGVuZCB0aGUgcmVhbCBtYXJrZXIgaXMgMjAgY20KICAgIHIgPSBzY2FsZV9mcm9tX21hcmtlcihbaW1nXSwgd3AsIG1hcmtlcl9jbSkKICAgICMgbWFya2VyIGVkZ2UgPSBweCBwaXhlbHMgPSBweCprIHVuaXRzOyB0cnVlIHNjYWxlID0gMjAgLyAocHgqaykKICAgIGV4cGVjdGVkID0gbWFya2VyX2NtIC8gKHB4ICogaykKICAgIGVyciA9IGFicyhyWyJzY2FsZV9jbV9wZXJfdW5pdCJdIC0gZXhwZWN0ZWQpIC8gZXhwZWN0ZWQgKiAxMDAKICAgIHByaW50KGYiZGV0ZWN0ZWQgbj17clsnbiddfSAgc2NhbGU9e3JbJ3NjYWxlX2NtX3Blcl91bml0J106LjRmfSAgZXhwZWN0ZWQ9e2V4cGVjdGVkOi40Zn0gIGVycj17ZXJyOi4yZn0lIikKICAgIHByaW50KCJBUlVDTyBTRUxGLVRFU1Q6IiwgIlBBU1MiIGlmIGVyciA8IDIgZWxzZSAiRkFJTCIpCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIF9zZWxmdGVzdCgpCiAgICBwcmludCgiZ2VuZXJhdGVkOiIsIGdlbmVyYXRlX21hcmtlcnMob3V0X2Rpcj1vcy5wYXRoLmpvaW4ob3MucGF0aC5kaXJuYW1lKF9fZmlsZV9fKSwgIm1hcmtlcnMiKSkpCg==").decode("utf-8"))
from estimate_volume import estimate_points
import aruco_tools

# >>> set this to the MEASURED edge length (cm) of your printed ArUco marker <<<
# (if you are not using a marker yet, leave it - scale then falls back to the rim)
MARKER_CM = 20.0

from vggt.models.vggt import VGGT
from vggt.utils.load_fn import load_and_preprocess_images
from vggt.utils.pose_enc import pose_encoding_to_extri_intri
from vggt.utils.geometry import unproject_depth_map_to_point_map
import gradio as gr

print("loading VGGT model (once)...")
MODEL = VGGT.from_pretrained("facebook/VGGT-1B").to("cuda").eval()

def extract_frames(video, n=40, out="frames"):
    if os.path.exists(out): shutil.rmtree(out)
    os.makedirs(out)
    cap = cv2.VideoCapture(video); total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    if total <= 0:
        total = 0
        while cap.grab(): total += 1
        cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
    slot = total / n; k = 0
    for i in range(n):
        lo, hi = int(i * slot), int((i + 1) * slot); best = None
        for idx in np.linspace(lo, max(lo, hi - 1), 5).astype(int):
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx)); ok, fr = cap.read()
            if not ok: continue
            sc = cv2.Laplacian(cv2.cvtColor(fr, cv2.COLOR_BGR2GRAY), cv2.CV_64F).var()
            if best is None or sc > best[1]: best = (idx, sc, fr)
        if best:
            cv2.imwrite(f"{out}/f_{k:03d}.jpg", best[2], [cv2.IMWRITE_JPEG_QUALITY, 95]); k += 1
    cap.release(); return sorted(glob.glob(f"{out}/*.jpg"))

def reconstruct(paths):
    dt = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
    imgs = load_and_preprocess_images(paths).to("cuda")
    with torch.no_grad(), torch.cuda.amp.autocast(dtype=dt):
        pred = MODEL(imgs)
    def gk(d, *ks):
        for kk in ks:
            if kk in d: return d[kk]
        raise KeyError(ks)
    extr, intr = pose_encoding_to_extri_intri(gk(pred, "pose_enc"), imgs.shape[-2:])
    depth = gk(pred, "depth", "depth_map"); conf = gk(pred, "depth_conf", "point_conf", "depth_confidence")
    wp = np.asarray(unproject_depth_map_to_point_map(depth.squeeze(0), extr.squeeze(0), intr.squeeze(0)))  # [N,H,W,3]
    cf = conf.squeeze(0).float().cpu().numpy()                    # [N,H,W]

    aruco_scale = None                                            # metric scale from markers (optional)
    try:
        im = imgs.detach().float().cpu().numpy().transpose(0, 2, 3, 1)          # [N,H,W,3]
        u8 = [((a - a.min()) / (a.max() - a.min() + 1e-9) * 255).astype("uint8") for a in im]
        rr = aruco_tools.scale_from_marker(u8, wp, MARKER_CM)
        if rr:
            aruco_scale = rr["scale_cm_per_unit"]; print("ArUco metric scale:", rr)
    except Exception as e:
        print("ArUco scale skipped:", e)

    w = wp.reshape(-1, 3); cff = cf.reshape(-1)
    w = w[(cff >= np.quantile(cff, 0.5)) & np.isfinite(w).all(1)]
    if len(w) > 300000:
        w = w[np.random.default_rng(0).choice(len(w), 300000, replace=False)]
    return w, aruco_scale

def process(video):
    if not video:
        return "<p>Please upload a kiln video.</p>", None, None
    try:
        paths = extract_frames(video)
        world, aruco_scale = reconstruct(paths)
        res = estimate_points(world, rim_radius_cm=75.0, scale_cm_per_unit=aruco_scale,
                              views_png="views.png", heatmap_png="heatmap.png")
    except Exception as e:
        return f"<p style='color:#b00'>Could not process this video: {e}</p>", None, None
    V, Vf = res["volume_L"], res["volume_L_flatfill"]
    fill = max(0, min(100, res["fill_pct"]))
    conf = res.get("confidence", "?")
    ccol = {"good": "#2e7d33", "medium": "#c60", "low": "#c0392b", "unreliable": "#c0392b"}.get(conf, "#666")
    warns = "".join(f"<li>{w}</li>" for w in res.get("warnings", []))
    warn_html = f"<ul style='color:#c60;margin:6px 0 0;padding-left:18px;font-size:13px'>{warns}</ul>" if warns else ""
    html = f"""<div style="font-family:system-ui,Segoe UI,Arial">
      <div style="font-size:13px;letter-spacing:1px;color:#888">ESTIMATED BIOCHAR VOLUME</div>
      <div style="font-size:54px;font-weight:800;color:#2e7d33;line-height:1">{V:,.0f} L</div>
      <div style="color:#555;margin-top:4px">&#8776; {res['weight_kg']:,.0f} kg &middot; {V/1000:.2f} m&sup3;</div>
      <div style="margin-top:12px;font-size:13px;color:#888">FILL &mdash; {fill:.0f}% of a ~1000 L kiln (height {res['fill_height_cm']:.0f} cm)</div>
      <div style="height:16px;background:#eee;border-radius:9px;overflow:hidden;margin-top:4px">
        <div style="height:100%;width:{fill:.0f}%;background:linear-gradient(90deg,#2d7ef7,#4fe08a)"></div></div>
      <div style="margin-top:12px">cross-check {Vf:,.0f} L &middot; scale from {res.get('scale_source','?')}</div>
      <div style="margin-top:10px;font-weight:700;color:{ccol}">Self-check: {conf.upper()}</div>
      {warn_html}
    </div>"""
    return html, "views.png", "heatmap.png"

demo = gr.Interface(
    fn=process,
    inputs=gr.Video(label="Upload a slow-orbit kiln video"),
    outputs=[gr.HTML(label="Result"),
             gr.Image(label="3-D reconstruction (top = rim disk, side = cone)"),
             gr.Image(label="Biochar depth heatmap (volume = sum)")],
    title="Kon-Tiki Biochar Volume - Video Dashboard",
    description="Upload a slow orbit video of the biochar-filled kiln. It extracts frames, reconstructs the 3-D shape on GPU, and returns the biochar volume. Follow the capture SOP for best accuracy.")
print("launching dashboard - a public https://....gradio.live link will appear below")
demo.launch(share=True)


### ArUco marker = true scale (recommended)
The result shows **"scale from marker"** or **"scale from rim (assumed)"**. To get the size
read from the *video itself* (not assumed), use ArUco markers:
1. Print the markers (`video_volume/markers/aruco_*.png`) on **matte** paper.
2. **Measure the black square's real edge with a ruler** and set `MARKER_CM` in Cell 1 to that value (cm).
3. Lay **3–4 markers flat on the ground** around the kiln, ~90° apart, different IDs (so one is always visible), **after quenching**.
4. Record the slow orbit as usual — the dashboard detects them and scales metrically.
No markers? It falls back to the assumed rim (fine for a standard Kon-Tiki 1000).

### Notes
- The **public link** works from any device for ~72 h while this cell runs.
- Keep this Colab tab open; closing it stops the dashboard. Re-run the cell to restart.
- For an **always-on** dashboard (no Colab), deploy the same app to a **GPU host**
  (Hugging Face Spaces GPU / Modal) — ask and I'll provide `gradio_app.py` + steps.
